# 03 - Adversarial Attacks

Evaluate the vulnerability of EEGNet under FGSM and PGD attacks.

In [ ]:
import sys
sys.path.append('..')
import torch
import matplotlib.pyplot as plt
from src.models.eegnet import EEGNet
from src.data.dataset import get_dataloaders
from experiments.run_experiment import train_base_model, evaluate_model
from src.attacks.fgsm import fgsm_attack
from src.attacks.pgd import pgd_attack

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, test_loader = get_dataloaders('../data/raw/', 1, batch_size=32)

model = EEGNet().to(device)
model = train_base_model(model, train_loader, epochs=10, lr=0.001, device=device) # Quick train

clean_acc = evaluate_model(model, test_loader, device=device)
print(f"Clean accuracy: {clean_acc:.2f}%")

epsilons = [0.01, 0.05, 0.1, 0.2, 0.3]
fgsm_accs = []
for eps in epsilons:
    acc = evaluate_model(model, test_loader, device=device, attack_fn=fgsm_attack, epsilon=eps)
    fgsm_accs.append(acc)
    print(f"FGSM eps={eps} accuracy: {acc:.2f}%")

plt.plot(epsilons, fgsm_accs, marker='o')
plt.title('Accuracy vs FGSM Epsilon')
plt.xlabel('Epsilon')
plt.ylabel('Accuracy (%)')
plt.show()